# 09_B_TEST_TUNING — PERCOBAAN 3 (Round 3 Refinement)

##  BASIS KEPUTUSAN DARI PERCOBAAN 2

### Temuan Pattern dari 78 Eksperimen Round 2:

1. **Recipe EXP_K menang mutlak** (F2=0.668, best_epoch=23, legitimate):
   - cosine_warm **T_0=20** (long cycle)
   - **CE** loss (bukan Focal)
   - **Label Smoothing = 0.0**
   - **WD = 1e-4** (sweet spot: 5e-4 terlalu besar, 5e-5 terlalu kecil)
   - **LR = 2e-4**
   - max_epochs=120, patience=30 (runway panjang)
   - hidden_dim=256, no augmentation

2. **Focal Loss GAGAL di SWIN** (γ=2 → F2=0.54, γ=1.5 → F2=0.65 di 5 epoch only)
   → Hipotesis Bab 4: label noise NTHU-DDD2 tinggi, focal over-fokus ke outlier

3. **Temporal Augmentation HANCUR di SWIN** (F2=0.46)
   → Frame drop merusak sequence continuity di pre-extracted features

4. **BiLSTM KALAH di SWIN** (avg F2: LSTM=0.556, BiLSTM=0.543)
   → BiLSTM butuh WD lebih ketat (3e-4) atau hidden lebih besar

5. **Degenerate solutions terdeteksi** (best_epoch=1, OneCycleLR + F2 selection):
   - CNN_BASIC_BILSTM_EXP_E: 78% prediksi=drowsy (trivial recall-bias)
   - CNN_BASIC_LSTM_EXP_E, VGG19_LSTM_EXP_E: sama
   → Ditambahkan DEGENERATE flag di Cell 11, bukan didiskualifikasi

##  RENCANA ROUND 3 — Exploit Recipe EXP_K

Hanya variasi **1 dimensi** per eksperimen (ablation discipline).
Target: F2 Drowsy ≥ 0.70 (vs current 0.668), FN ≤ 130 (vs 155).

| Exp | Variasi | Hipotesis |
|---|---|---|
| EXP_K2_T30 | T_0=30, max_epochs=150 | Cycle lebih panjang |
| EXP_K2_LR1e4 | LR=1e-4 | Fine-grained convergence |
| EXP_K2_H384 | hidden_dim=384 | Sweet spot kapasitas |
| EXP_K2_H512_WD3 | hidden=512 + WD=3e-4 | Kapasitas naik + WD kompensasi |
| EXP_K2_DROP4 | lstm_dropout=0.4, fc_dropout=0.5 | Regularisasi lebih kuat |
| EXP_K2_FOCAL15 | Focal γ=1.5 (sisanya EXP_K) | Recreate EXP_I dengan runway penuh |
| EXP_K2_LAYER3 | num_layers=3 | Depth temporal lebih dalam |
| EXP_K2_BI_H512_WD3 | BiLSTM + h=512 + WD=3e-4 | Selamatkan BiLSTM |
| EXP_K2_WARMUP | LinearWarmup 5ep + cosine_warm | Hindari epoch awal noisy |
| EXP_K2_AGG | T_0=30 + h=384 + drop=0.4 | Compound best-of variasi |
| VGG19_LSTM_EXP_K | Recipe EXP_K pada VGG19 | Sanity check backbone |

##  Target Akhir Skripsi
- F2 Drowsy ≥ 0.70 (dengan Precision ≥ 0.52)
- F1 Macro ≥ 0.65
- Recall Drowsy ≥ 0.77
- FN Drowsy ≤ 130
- Best Epoch > 5 (legitimate, bukan degenerate)


#  Import & Konfigurasi Global

In [1]:
# ============================================================
# CELL 1 — Import & Konfigurasi Global (Percobaan 3)
# ============================================================
# CATATAN:
#   SAVE_DIR sama dengan Percobaan 2 (tuning_results_nthuddd2).
#   Experiment ID baru (EXP_K2_*) tidak akan konflik dengan hasil P2.
#   Cell 11/12 akan membaca SEMUA result termasuk P2 — ranking gabungan.
# ============================================================

import os, gc, json, time, random, warnings
from copy import deepcopy
from itertools import product

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import (
    CosineAnnealingWarmRestarts,
    OneCycleLR,
    ReduceLROnPlateau,
    LambdaLR,
)
from sklearn.metrics import (
    f1_score, fbeta_score, accuracy_score, precision_score,
    recall_score, confusion_matrix, roc_auc_score,
)
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import pandas as pd

warnings.filterwarnings("ignore")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# ── SEED ──────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

# ── DEVICE ────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── PATH ──────────────────────────────────────────────────────
BASE_DIR  = r"C:\kuliah-sementara\SKRIPSI"
SEQ_DIR   = os.path.join(BASE_DIR, "sequences_nthuddd2")
SAVE_DIR  = os.path.join(BASE_DIR, "tuning_results_nthuddd2")   # ← sama dengan P2
os.makedirs(SAVE_DIR, exist_ok=True)

# ── DATA CONFIG ───────────────────────────────────────────────
MODELS      = ["CNN_BASIC", "MOBILENET", "VGG19", "SWIN"]
SPLITS      = ["train", "val", "test"]
INPUT_DIM   = 512
WINDOW_SIZE = 30
LABEL_MAP   = {0: "drowsy", 1: "notdrowsy"}
NUM_CLASSES = 2
NUM_WORKERS = 0
DROWSY_CLASS_IDX    = 0
NOTDROWSY_CLASS_IDX = 1

print("=" * 65)
print("  09_B_TEST_TUNING — PERCOBAAN 3 (Round 3 Refinement)")
print("=" * 65)
print(f"  Device      : {DEVICE}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU         : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM Total  : {props.total_memory/1e9:.1f} GB")
print(f"  SAVE_DIR    : {SAVE_DIR}")
print(f"  Mode        : Round 3 — Exploit EXP_K recipe")
print("=" * 65)
print("✓ CELL 1 selesai.")


  09_B_TEST_TUNING — PERCOBAAN 3 (Round 3 Refinement)
  Device      : cuda
  GPU         : NVIDIA GeForce RTX 3060
  VRAM Total  : 12.9 GB
  SAVE_DIR    : C:\kuliah-sementara\SKRIPSI\tuning_results_nthuddd2
  Mode        : Round 3 — Exploit EXP_K recipe
✓ CELL 1 selesai.


# Validasi File Sekuens

In [2]:
# ============================================================
# CELL 2 — Validasi File Sekuens Input
# ============================================================
# Pastikan semua 12 file .pt dari notebook 08 tersedia.
# Sama persis dengan file 09 asli.
# ============================================================

print("[VALIDASI] Mengecek file sekuens input...\n")

all_ok = True
for model in MODELS:
    for split in SPLITS:
        fname = f"{model}_Seq_{split.capitalize()}_NTHUD.pt"
        fpath = os.path.join(SEQ_DIR, model, fname)
        exists = os.path.exists(fpath)
        if exists:
            data    = torch.load(fpath, map_location="cpu", weights_only=False)
            n_seq   = data["features"].shape[0]
            n_drow  = int((data["labels"] == 0).sum())
            n_notd  = int((data["labels"] == 1).sum())
            balance = n_drow / n_seq * 100
            print(f"  [OK] {fname}")
            print(f"       Shape={list(data['features'].shape)} | "
                  f"drowsy={n_drow:,} ({balance:.1f}%) | notdrowsy={n_notd:,}")
        else:
            print(f"  [MISSING] {fname}")
            all_ok = False

print()
if not all_ok:
    raise FileNotFoundError(
        "Beberapa file .pt tidak ditemukan!\n"
        "Pastikan notebook 08 sudah dijalankan."
    )
print("✓ Semua file sekuens tersedia. Siap eksperimen tuning.")

[VALIDASI] Mengecek file sekuens input...

  [OK] CNN_BASIC_Seq_Train_NTHUD.pt
       Shape=[8053, 30, 512] | drowsy=4,502 (55.9%) | notdrowsy=3,551
  [OK] CNN_BASIC_Seq_Val_NTHUD.pt
       Shape=[3634, 30, 512] | drowsy=2,100 (57.8%) | notdrowsy=1,534
  [OK] CNN_BASIC_Seq_Test_NTHUD.pt
       Shape=[1302, 30, 512] | drowsy=548 (42.1%) | notdrowsy=754
  [OK] MOBILENET_Seq_Train_NTHUD.pt
       Shape=[8053, 30, 512] | drowsy=4,502 (55.9%) | notdrowsy=3,551
  [OK] MOBILENET_Seq_Val_NTHUD.pt
       Shape=[3634, 30, 512] | drowsy=2,100 (57.8%) | notdrowsy=1,534
  [OK] MOBILENET_Seq_Test_NTHUD.pt
       Shape=[1302, 30, 512] | drowsy=548 (42.1%) | notdrowsy=754
  [OK] VGG19_Seq_Train_NTHUD.pt
       Shape=[8053, 30, 512] | drowsy=4,502 (55.9%) | notdrowsy=3,551
  [OK] VGG19_Seq_Val_NTHUD.pt
       Shape=[3634, 30, 512] | drowsy=2,100 (57.8%) | notdrowsy=1,534
  [OK] VGG19_Seq_Test_NTHUD.pt
       Shape=[1302, 30, 512] | drowsy=548 (42.1%) | notdrowsy=754
  [OK] SWIN_Seq_Train_NTHUD.pt
     

# Dataset + DataLoader + Temporal Augmentation

In [3]:
# ============================================================
# CELL 3 — Dataset Class + DataLoader
# ============================================================
# PERUBAHAN vs File 09:
#   + SequenceDatasetAugmented: mendukung temporal augmentation
#     - Frame Dropping (1-2 frame, copy neighbor)
#     - Gaussian Noise kecil (std=0.01)
#   Augmentasi HANYA aktif pada training set (augment=True).
#   Val/Test tetap bersih tanpa augmentasi.
# ============================================================

class SequenceDatasetAugmented(Dataset):
    """
    Dataset wrapper dengan optional Temporal Augmentation untuk training.

    Temporal Augmentation:
    1. Frame Dropping  : Hapus 1-2 frame secara acak dari 30 frame,
                         gantikan dengan copy frame tetangga terdekat.
                         Tujuan: simulasi kamera lag / dropped frame.
    2. Gaussian Noise  : Tambahkan noise N(0, 0.01) ke semua fitur.
                         Tujuan: membuat LSTM lebih robust terhadap
                         variasi kecil pada fitur CNN.

    Args:
        features    : Tensor [N, T, D]
        labels      : Tensor [N]
        mean / std  : Statistik dari training set (Z-Score normalisasi)
        augment     : True = aktifkan augmentasi (hanya untuk train)
        aug_prob_fd : Probabilitas frame dropping per sampel
        aug_prob_gn : Probabilitas gaussian noise per sampel
        n_drop_max  : Maksimum frame yang di-drop (1 atau 2)
        noise_std   : Standar deviasi Gaussian noise
    """
    def __init__(
        self,
        features    : torch.Tensor,
        labels      : torch.Tensor,
        mean        : torch.Tensor = None,
        std         : torch.Tensor = None,
        augment     : bool  = False,
        aug_prob_fd : float = 0.3,   # 30% chance frame dropping
        aug_prob_gn : float = 0.3,   # 30% chance gaussian noise
        n_drop_max  : int   = 1,     # Max 1 frame di-drop (konservatif)
        noise_std   : float = 0.01,  # Noise sangat kecil
    ):
        self.features    = features.float()
        self.labels      = labels.long()
        self.augment     = augment
        self.aug_prob_fd = aug_prob_fd
        self.aug_prob_gn = aug_prob_gn
        self.n_drop_max  = n_drop_max
        self.noise_std   = noise_std

        # Z-Score Normalisasi (dari train stats)
        if mean is not None and std is not None:
            std_safe = std.clamp(min=1e-8)
            self.features = (self.features - mean) / std_safe

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        feat = self.features[idx].clone()  # [T=30, D=512]

        if self.augment:
            # ── Augmentasi 1: Frame Dropping ──────────────────
            # Hapus 1–n_drop_max frame secara acak, gantikan
            # dengan copy frame tetangga (copy-paste nearest)
            if random.random() < self.aug_prob_fd:
                n_drop = random.randint(1, self.n_drop_max)
                T = feat.shape[0]
                drop_idx = sorted(
                    random.sample(range(T), n_drop), reverse=True
                )
                for di in drop_idx:
                    # Pilih tetangga: sebelum atau sesudah
                    neighbor = di - 1 if di > 0 else di + 1
                    neighbor = max(0, min(neighbor, T - 1))
                    feat[di] = feat[neighbor].clone()

            # ── Augmentasi 2: Gaussian Noise ──────────────────
            # Noise kecil untuk mencegah LSTM "menghafal" pola
            if random.random() < self.aug_prob_gn:
                noise = torch.randn_like(feat) * self.noise_std
                feat = feat + noise

        return feat, self.labels[idx]


def load_split_data(model_name: str, split: str):
    """Load file .pt untuk satu backbone + satu split."""
    fname = f"{model_name}_Seq_{split.capitalize()}_NTHUD.pt"
    fpath = os.path.join(SEQ_DIR, model_name, fname)
    data  = torch.load(fpath, map_location="cpu", weights_only=False)
    return data["features"].float(), data["labels"].long()


def compute_train_stats(features: torch.Tensor):
    """Hitung mean & std dari training set untuk Z-Score normalisasi."""
    N, T, D = features.shape
    flat    = features.reshape(-1, D)
    return flat.mean(dim=0), flat.std(dim=0)


def build_dataloaders(
    model_name   : str,
    batch_size   : int  = 64,
    use_aug      : bool = False,
    aug_prob_fd  : float = 0.3,
    aug_prob_gn  : float = 0.3,
    n_drop_max   : int  = 1,
    noise_std    : float = 0.01,
):
    """
    Build train/val/test DataLoader untuk satu backbone.
    Normalisasi dari train set, augmentasi hanya pada train jika use_aug=True.

    Returns:
        loaders      : dict {train, val, test}
        class_weight : Tensor untuk weighted loss
        norm_stats   : dict {mean, std}
    """
    # Load semua split
    train_feat, train_lab = load_split_data(model_name, "train")
    val_feat,   val_lab   = load_split_data(model_name, "val")
    test_feat,  test_lab  = load_split_data(model_name, "test")

    # Normalisasi dari train saja (cegah data leakage)
    mean, std = compute_train_stats(train_feat)

    # Dataset
    train_ds = SequenceDatasetAugmented(
        train_feat, train_lab, mean, std,
        augment=use_aug,
        aug_prob_fd=aug_prob_fd,
        aug_prob_gn=aug_prob_gn,
        n_drop_max=n_drop_max,
        noise_std=noise_std,
    )
    val_ds  = SequenceDatasetAugmented(val_feat,  val_lab,  mean, std, augment=False)
    test_ds = SequenceDatasetAugmented(test_feat, test_lab, mean, std, augment=False)

    # Class Weight: w_c = N_total / (num_classes × N_c)
    label_counts = Counter(train_lab.numpy())
    n_total      = len(train_lab)
    weights      = torch.tensor([
        n_total / (NUM_CLASSES * label_counts[c])
        for c in range(NUM_CLASSES)
    ], dtype=torch.float32)

    pin = (DEVICE.type == "cuda")
    loaders = {
        "train": DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=pin),
        "val"  : DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=pin),
        "test" : DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=pin),
    }
    norm_stats = {"mean": mean, "std": std}

    print(f"  [DataLoader] {model_name} | batch={batch_size} | aug={'ON' if use_aug else 'OFF'}")
    print(f"  [Norm]       mean={mean.mean():.4f}  std={std.mean():.4f}")
    print(f"  [ClassW]     drowsy={weights[0]:.4f}  notdrowsy={weights[1]:.4f}")

    return loaders, weights, norm_stats


print("✓ CELL 3 — Dataset + DataLoader + Augmentasi terdefinisi.")

✓ CELL 3 — Dataset + DataLoader + Augmentasi terdefinisi.


# Arsitektur Model (Fleksibel + GELU)

In [4]:
# ============================================================
# CELL 4 — Arsitektur Model
# ============================================================
# PERUBAHAN vs File 09:
#   + Aktivasi FC bisa dipilih: ReLU / GELU / SiLU
#     - GELU direkomendasikan untuk Swin (konsisten dengan
#       aktivasi internal Swin Transformer)
#   + LSTM hidden_dim bisa 256 atau 512 (tunable)
#   + num_layers bisa 2 atau 3 (tunable)
#   + Semua parameter dipass lewat config dict (bukan global)
# ============================================================

class AdditiveAttention(nn.Module):
    """
    Additive (Bahdanau-style) Attention untuk output LSTM.
    Input  : [B, T, H]
    Output : context [B, H], weights [B, T]
    """
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, hidden_dim)
        self.v    = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_out: torch.Tensor):
        energy  = torch.tanh(self.attn(lstm_out))       # [B, T, H]
        scores  = self.v(energy).squeeze(-1)             # [B, T]
        weights = torch.softmax(scores, dim=-1)          # [B, T]
        context = torch.bmm(weights.unsqueeze(1), lstm_out).squeeze(1)  # [B, H]
        return context, weights


# ── Activation Factory ────────────────────────────────────────
_ACTIVATION_MAP = {
    "relu" : nn.ReLU,
    "gelu" : nn.GELU,
    "silu" : nn.SiLU,
    "elu"  : nn.ELU,
}

def get_activation(name: str) -> nn.Module:
    name = name.lower()
    if name not in _ACTIVATION_MAP:
        raise ValueError(f"Aktivasi '{name}' tidak dikenal. Pilih: {list(_ACTIVATION_MAP)}")
    return _ACTIVATION_MAP[name]()


class DrowsinessLSTM(nn.Module):
    """
    Arsitektur utama: Stacked LSTM/BiLSTM + Attention + FC Classifier.

    Perbedaan vs File 09:
    - Aktivasi FC bisa dipilih via `fc_activation` ('relu'/'gelu'/'silu')
    - hidden_dim, num_layers dapat diatur dari config eksperimen
    """
    def __init__(
        self,
        input_dim     : int   = 512,
        hidden_dim    : int   = 256,
        num_layers    : int   = 2,
        num_classes   : int   = 2,
        bidirectional : bool  = False,
        use_attention : bool  = True,
        lstm_dropout  : float = 0.3,
        fc_dropout    : float = 0.5,
        fc_activation : str   = "relu",   # ← BARU: pilihan aktivasi
    ):
        super().__init__()
        self.bidirectional = bidirectional
        self.use_attention = use_attention
        self.hidden_dim    = hidden_dim
        self.num_layers    = num_layers
        num_dir            = 2 if bidirectional else 1
        lstm_out_dim       = hidden_dim * num_dir

        # ── Layer Normalisasi Input ──────────────────────────
        self.input_norm = nn.LayerNorm(input_dim)

        # ── LSTM / BiLSTM ────────────────────────────────────
        self.lstm = nn.LSTM(
            input_size    = input_dim,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            bidirectional = bidirectional,
            dropout       = lstm_dropout if num_layers > 1 else 0.0,
        )

        # ── Attention ────────────────────────────────────────
        if use_attention:
            self.attention = AdditiveAttention(lstm_out_dim)
        else:
            self.attention = None
        classifier_in = lstm_out_dim

        # ── FC Classifier (dengan pilihan aktivasi) ──────────
        act = get_activation(fc_activation)
        self.classifier = nn.Sequential(
            nn.Dropout(fc_dropout),
            nn.Linear(classifier_in, 128),
            act,                            # ← GELU untuk Swin
            nn.Dropout(fc_dropout * 0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor):
        x        = self.input_norm(x)                          # [B, T, D]
        lstm_out, (h_n, _) = self.lstm(x)                     # [B, T, H*dir]

        if self.use_attention:
            context, attn_w = self.attention(lstm_out)         # [B, H*dir]
        else:
            context = lstm_out[:, -1, :]
            attn_w  = None

        logits = self.classifier(context)                      # [B, C]
        return logits, attn_w


def build_model_from_cfg(cfg: dict) -> DrowsinessLSTM:
    """
    Bangun model dari dictionary konfigurasi eksperimen.
    Contoh cfg: {hidden_dim:256, num_layers:2, lstm_dropout:0.3,
                 fc_dropout:0.5, fc_activation:'gelu',
                 bidirectional:False, use_attention:True}
    """
    return DrowsinessLSTM(
        input_dim     = INPUT_DIM,
        hidden_dim    = cfg.get("hidden_dim",    256),
        num_layers    = cfg.get("num_layers",    2),
        num_classes   = NUM_CLASSES,
        bidirectional = cfg.get("bidirectional", False),
        use_attention = cfg.get("use_attention", True),
        lstm_dropout  = cfg.get("lstm_dropout",  0.3),
        fc_dropout    = cfg.get("fc_dropout",    0.5),
        fc_activation = cfg.get("fc_activation", "relu"),
    ).to(DEVICE)


# ── Sanity check arsitektur ───────────────────────────────────
print("[TEST] Memverifikasi semua kombinasi arsitektur...")
for act in ["relu", "gelu", "silu"]:
    for hidden in [256, 512]:
        cfg_test = dict(
            hidden_dim=hidden, num_layers=2, lstm_dropout=0.3,
            fc_dropout=0.5, fc_activation=act, bidirectional=False, use_attention=True
        )
        m   = build_model_from_cfg(cfg_test)
        x   = torch.randn(2, WINDOW_SIZE, INPUT_DIM).to(DEVICE)
        out, _ = m(x)
        assert out.shape == (2, NUM_CLASSES), f"Shape error: {out.shape}"
        del m, x, out
torch.cuda.empty_cache()
print("✓ Semua kombinasi arsitektur valid.")
print("✓ CELL 4 — Arsitektur Model terdefinisi.")

[TEST] Memverifikasi semua kombinasi arsitektur...
✓ Semua kombinasi arsitektur valid.
✓ CELL 4 — Arsitektur Model terdefinisi.


# Loss Functions (Focal Loss + Factory)

In [5]:
# ============================================================
# CELL 5 — Loss Functions
# ============================================================
# PERUBAHAN vs File 09:
#   + FocalLoss: down-weight easy examples, fokus ke hard ones
#     Referensi: Lin et al. (2017) "Focal Loss for Dense Object Detection"
#   + loss_factory(): pilih loss berdasarkan string config
#
# Kapan pakai Focal Loss?
# - Dataset ini: drowsy 42.1% vs notdrowsy 57.9% di test set
# - Imbalance moderat (bukan ekstrem), jadi Focal loss hanya
#   membantu jika dikombinasikan dengan class_weight
# - Untuk Swin: patut dicoba karena F1 drowsy masih rendah
# ============================================================

class FocalLoss(nn.Module):
    """
    Focal Loss = -α_t * (1 - p_t)^γ * log(p_t)

    Args:
        gamma        : Focusing parameter. 0 = CE biasa.
                       Saran: gamma=1.0 (moderat) atau gamma=2.0 (agresif)
        weight       : Class weight tensor (dari build_dataloaders)
        label_smoothing: Smoothing ε (0.0 - tidak aktif, 0.05 - ringan)
        reduction    : 'mean' (default)

    Note:
        Focal Loss + class_weight = kombinasi yang paling powerful
        untuk imbalanced binary classification di sistem safety-critical.
    """
    def __init__(
        self,
        gamma           : float = 2.0,
        weight          : torch.Tensor = None,
        label_smoothing : float = 0.0,
        reduction       : str   = "mean",
    ):
        super().__init__()
        self.gamma           = gamma
        self.weight          = weight
        self.label_smoothing = label_smoothing
        self.reduction       = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        # CrossEntropy dengan label smoothing (per-sample, reduction='none')
        ce_loss = F.cross_entropy(
            logits, targets,
            weight          = self.weight,
            label_smoothing = self.label_smoothing,
            reduction       = "none",
        )
        # p_t: probabilitas untuk kelas yang benar
        pt          = torch.exp(-ce_loss)
        focal_loss  = ((1.0 - pt) ** self.gamma) * ce_loss

        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        return focal_loss


def build_loss(cfg: dict, class_weight: torch.Tensor) -> nn.Module:
    """
    Factory function untuk membangun loss dari konfigurasi eksperimen.

    cfg keys:
        loss_type       : 'ce' (CrossEntropy) atau 'focal'
        label_smoothing : float (0.0–0.1)
        focal_gamma     : float (1.0 atau 2.0, hanya untuk focal)
    """
    loss_type       = cfg.get("loss_type",        "ce")
    label_smoothing = cfg.get("label_smoothing",  0.0)
    focal_gamma     = cfg.get("focal_gamma",      2.0)
    cw              = class_weight.to(DEVICE)

    if loss_type == "focal":
        return FocalLoss(
            gamma           = focal_gamma,
            weight          = cw,
            label_smoothing = label_smoothing,
        ).to(DEVICE)
    else:  # "ce"
        return nn.CrossEntropyLoss(
            weight          = cw,
            label_smoothing = label_smoothing,
        ).to(DEVICE)


# ── Tes loss functions ────────────────────────────────────────
_dummy_logits  = torch.randn(4, 2).to(DEVICE)
_dummy_targets = torch.tensor([0, 1, 0, 1]).to(DEVICE)
_dummy_weight  = torch.tensor([0.89, 1.13]).to(DEVICE)

_fl = FocalLoss(gamma=2.0, weight=_dummy_weight)(_dummy_logits, _dummy_targets)
_ce = nn.CrossEntropyLoss(weight=_dummy_weight, label_smoothing=0.05)(
    _dummy_logits, _dummy_targets
)
print(f"  FocalLoss test  : {_fl.item():.4f}")
print(f"  CrossEntropy test: {_ce.item():.4f}")
del _dummy_logits, _dummy_targets, _dummy_weight, _fl, _ce
print("✓ CELL 5 — Loss Functions terdefinisi.")

  FocalLoss test  : 0.7103
  CrossEntropy test: 1.1207
✓ CELL 5 — Loss Functions terdefinisi.


# LR Range Test Utility

In [6]:
# ============================================================
# CELL 6 — LR Range Test Utility
# ============================================================
# Implementasi dari Smith (2017) arXiv:1803.09820
#
# Cara kerja:
# - LR dinaikkan secara eksponensial dari start_lr → end_lr
# - Catat loss di setiap step
# - Plot: pilih LR di area SEBELUM loss mulai naik
#         (biasanya titik kemiringan paling curam ke bawah)
#
# Hasil LR Range Test digunakan sebagai max_lr untuk:
# - CosineAnnealingWarmRestarts
# - OneCycleLR
# ============================================================

def lr_range_test(
    backbone_name : str,
    cfg           : dict,
    start_lr      : float = 1e-7,
    end_lr        : float = 1e-1,
    num_iter      : int   = 150,
    batch_size    : int   = 64,
    smooth_beta   : float = 0.9,    # Exponential smoothing untuk kurva
    save_plot     : bool  = True,
):
    """
    Jalankan LR Range Test untuk satu backbone + konfigurasi.

    Returns:
        best_lr : float — LR yang disarankan (di kemiringan terbesar)
        lrs     : list[float]
        losses  : list[float] (smoothed)
    """
    print(f"\n[LR Range Test] Backbone={backbone_name} | "
          f"LR {start_lr:.0e} → {end_lr:.0e} | {num_iter} iter")

    # Build data & model baru (fresh)
    loaders, class_weight, _ = build_dataloaders(
        backbone_name, batch_size=batch_size
    )
    model     = build_model_from_cfg(cfg)
    criterion = build_loss(cfg, class_weight)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=start_lr,
        weight_decay=cfg.get("weight_decay", 1e-4)
    )

    # Multiplier LR per iterasi
    mult = (end_lr / start_lr) ** (1.0 / num_iter)
    lrs, raw_losses, smooth_losses = [], [], []
    avg_loss, best_loss = 0.0, float("inf")

    model.train()
    train_iter = iter(loaders["train"])

    for i in range(num_iter):
        try:
            feats, labels = next(train_iter)
        except StopIteration:
            train_iter = iter(loaders["train"])
            feats, labels = next(train_iter)

        feats  = feats.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits, _ = model(feats)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        # Exponential smoothing
        avg_loss = smooth_beta * avg_loss + (1 - smooth_beta) * loss.item()
        smoothed = avg_loss / (1 - smooth_beta ** (i + 1))  # Bias correction

        lrs.append(optimizer.param_groups[0]["lr"])
        raw_losses.append(loss.item())
        smooth_losses.append(smoothed)

        # Stop jika loss meledak (4× loss minimum)
        if smoothed < best_loss:
            best_loss = smoothed
        if smoothed > 4 * best_loss:
            print(f"  Loss diverged di iterasi {i+1}. Berhenti.")
            break

        # Naikkan LR
        for pg in optimizer.param_groups:
            pg["lr"] *= mult

    # ── Temukan LR optimal: kemiringan paling curam ke bawah ──
    # Hitung gradien kurva smoothed loss, pilih titik minimum
    if len(smooth_losses) > 10:
        grad      = np.gradient(smooth_losses)
        min_grad_idx = np.argmin(grad[5:]) + 5  # Skip 5 iterasi awal
        best_lr   = lrs[min_grad_idx] / 10.0   # Sedikit lebih konservatif
    else:
        best_lr = 1e-4

    print(f"  → Saran max_lr : {best_lr:.2e} "
          f"(LR di kemiringan terbesar = {lrs[min_grad_idx]:.2e})")

    # ── Plot ──────────────────────────────────────────────────
    fig, ax = plt.subplots(1, 1, figsize=(9, 4))
    ax.plot(lrs[:len(smooth_losses)], smooth_losses, lw=2, color="#2563eb", label="Loss (smoothed)")
    ax.axvline(best_lr, color="#dc2626", linestyle="--", lw=1.5,
               label=f"Saran max_lr = {best_lr:.2e}")
    ax.set_xscale("log")
    ax.set_xlabel("Learning Rate (log scale)", fontsize=11)
    ax.set_ylabel("Loss", fontsize=11)
    ax.set_title(f"LR Range Test — {backbone_name}", fontsize=13, fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    if save_plot:
        plot_dir = os.path.join(SAVE_DIR, "lr_range_tests")
        os.makedirs(plot_dir, exist_ok=True)
        plot_path = os.path.join(plot_dir, f"lr_range_{backbone_name}.png")
        plt.savefig(plot_path, dpi=120, bbox_inches="tight")
        print(f"  Plot disimpan: {plot_path}")
    plt.show()

    # Cleanup
    del model, loaders, optimizer, criterion
    gc.collect()
    torch.cuda.empty_cache()

    return best_lr, lrs, smooth_losses


# ── Contoh cara pakai (jalankan sebelum CELL 9 jika ingin LR test) ──
# Contoh: best_lr_swin, _, _ = lr_range_test("SWIN", cfg_swin_base)
# Lalu masukkan best_lr_swin ke EXPERIMENT_GRID di CELL 9

print("✓ CELL 6 — LR Range Test terdefinisi.")
print("  Cara pakai: lr_range_test('SWIN', cfg_dict)")
print("  Jalankan ini SEBELUM CELL 9 untuk mendapatkan LR optimal per backbone.")

✓ CELL 6 — LR Range Test terdefinisi.
  Cara pakai: lr_range_test('SWIN', cfg_dict)
  Jalankan ini SEBELUM CELL 9 untuk mendapatkan LR optimal per backbone.


# Training Core (CosineWarmRestart + Cyclical Momentum)


In [7]:
# ============================================================
# CELL 7 — Training Core Functions (Percobaan 3)
# ============================================================
# PERUBAHAN vs Percobaan 2:
#   + Helper: build_linear_warmup_cosine_scheduler() untuk EXP_K2_WARMUP
#   Sisanya identik dengan P2 (F2 Drowsy metrics, scheduler bug fix).
# ============================================================

def _compute_metrics(preds, labels):
    """Metrik lengkap: F2 Drowsy sebagai utama."""
    preds  = np.asarray(preds)
    labels = np.asarray(labels)
    return {
        "acc"              : accuracy_score(labels, preds),
        "f1_macro"         : f1_score(labels, preds, average="macro", zero_division=0),
        "f2_drowsy"        : fbeta_score(
            labels, preds, beta=2.0, pos_label=DROWSY_CLASS_IDX,
            average="binary", zero_division=0
        ),
        "recall_drowsy"    : recall_score(
            labels, preds, pos_label=DROWSY_CLASS_IDX,
            average="binary", zero_division=0
        ),
        "precision_drowsy" : precision_score(
            labels, preds, pos_label=DROWSY_CLASS_IDX,
            average="binary", zero_division=0
        ),
    }


def train_epoch(model, loader, criterion, optimizer, scheduler=None,
                sched_type="per_epoch", max_grad_norm=1.0):
    """Satu epoch training. Returns dict metrik lengkap."""
    model.train()
    total_loss            = 0.0
    all_preds, all_labels = [], []

    for feats, labels in loader:
        feats  = feats.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits, _ = model(feats)
        loss      = criterion(logits, labels)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
        optimizer.step()

        if sched_type == "per_batch" and scheduler is not None:
            scheduler.step()

        total_loss += loss.item() * len(labels)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    metrics  = _compute_metrics(all_preds, all_labels)
    metrics["loss"] = avg_loss
    return metrics


@torch.no_grad()
def evaluate(model, loader, criterion):
    """Evaluasi model. Returns dict metrik + preds + labels."""
    model.eval()
    total_loss            = 0.0
    all_preds, all_labels = [], []

    for feats, labels in loader:
        feats  = feats.to(DEVICE)
        labels = labels.to(DEVICE)
        logits, _ = model(feats)
        loss = criterion(logits, labels)

        total_loss += loss.item() * len(labels)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    metrics  = _compute_metrics(all_preds, all_labels)
    metrics["loss"]   = avg_loss
    metrics["preds"]  = np.array(all_preds)
    metrics["labels"] = np.array(all_labels)
    return metrics


# ── NEW: LinearWarmup wrapper untuk EXP_K2_WARMUP ─────────────
class LinearWarmupCosineWarmRestarts:
    """
    Schedule hybrid:
      1. Epoch 0..warmup_epochs-1: LR linear dari 1e-6 → base_lr
      2. Epoch warmup_epochs..end: CosineAnnealingWarmRestarts
    Tujuan: hindari gradient noise di epoch-epoch awal yang bisa
    bikin model konvergen ke trivial/degenerate solution.
    """
    def __init__(self, optimizer, warmup_epochs, cosine_kwargs):
        self.optimizer     = optimizer
        self.warmup_epochs = warmup_epochs
        self.base_lrs      = [pg["lr"] for pg in optimizer.param_groups]
        self.cosine        = None          # lazy init setelah warmup
        self.cosine_kwargs = cosine_kwargs
        self.step_count    = 0

    def step(self, epoch=None):
        if epoch is None:
            epoch = self.step_count
        self.step_count = epoch

        if epoch < self.warmup_epochs:
            # Linear warmup
            warmup_ratio = (epoch + 1) / self.warmup_epochs
            for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
                pg["lr"] = 1e-6 + (base_lr - 1e-6) * warmup_ratio
        else:
            if self.cosine is None:
                self.cosine = CosineAnnealingWarmRestarts(
                    self.optimizer, **self.cosine_kwargs
                )
            self.cosine.step(epoch - self.warmup_epochs)


def build_optimizer_and_scheduler(model, cfg, loaders):
    """Factory: optimizer + scheduler dari cfg dict."""
    lr             = cfg.get("lr",             1e-3)
    weight_decay   = cfg.get("weight_decay",   1e-4)
    opt_type       = cfg.get("optimizer_type", "adam")
    sched_type     = cfg.get("scheduler_type", "cosine_warm")
    max_epochs     = cfg.get("max_epochs",     60)

    # ── Optimizer ─────────────────────────────────────────────
    if opt_type == "adamw":
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=lr, weight_decay=weight_decay,
            betas=(0.9, 0.999),
        )
    else:
        optimizer = torch.optim.Adam(
            model.parameters(), lr=lr, weight_decay=weight_decay,
        )

    steps_per_epoch = len(loaders["train"])

    # ── Scheduler ─────────────────────────────────────────────
    if sched_type == "cosine_warm":
        scheduler = CosineAnnealingWarmRestarts(
            optimizer,
            T_0    = cfg.get("T_0",    10),
            T_mult = cfg.get("T_mult", 2),
            eta_min= cfg.get("eta_min", 1e-6),
        )
        sched_step_type = "per_epoch"

    elif sched_type == "warmup_cosine":
        scheduler = LinearWarmupCosineWarmRestarts(
            optimizer,
            warmup_epochs = cfg.get("warmup_epochs", 5),
            cosine_kwargs = dict(
                T_0    = cfg.get("T_0",    10),
                T_mult = cfg.get("T_mult", 2),
                eta_min= cfg.get("eta_min", 1e-6),
            ),
        )
        sched_step_type = "per_epoch"

    elif sched_type == "onecycle":
        scheduler = OneCycleLR(
            optimizer,
            max_lr           = lr,
            epochs           = max_epochs,
            steps_per_epoch  = steps_per_epoch,
            pct_start        = cfg.get("pct_start",      0.3),
            anneal_strategy  = "cos",
            cycle_momentum   = True,
            base_momentum    = cfg.get("base_momentum",  0.85),
            max_momentum     = cfg.get("max_momentum",   0.95),
            div_factor       = cfg.get("div_factor",     25.0),
            final_div_factor = cfg.get("final_div_factor", 1e4),
        )
        sched_step_type = "per_batch"

    else:  # plateau
        scheduler = ReduceLROnPlateau(
            optimizer,
            mode      = "max",
            factor    = cfg.get("lr_factor",   0.5),
            patience  = cfg.get("lr_patience", 5),
            min_lr    = cfg.get("eta_min",     1e-6),
            threshold = 1e-4,
        )
        sched_step_type = "per_epoch_plateau"

    return optimizer, scheduler, sched_step_type


print("✓ CELL 7 Percobaan 3 — Training Core terdefinisi.")
print("  Scheduler baru: warmup_cosine (LinearWarmup + CosineWarmRestarts)")


✓ CELL 7 Percobaan 3 — Training Core terdefinisi.
  Scheduler baru: warmup_cosine (LinearWarmup + CosineWarmRestarts)


# Experiment Runner

In [8]:
# ============================================================
# CELL 8 — run_tuning_experiment() (v2 — F2 Drowsy Early Stopping)
# ============================================================
# PERUBAHAN v2:
#
#   1. EARLY STOPPING → F2-Score Drowsy (bukan F1 Macro)
#      Justifikasi: F2 mengoptimasi recall drowsy dengan penalti
#      precision yang ringan. Cocok dengan target skripsi safety-critical.
#
#   2. Tabel epoch menampilkan F1, F2, Recall_Drowsy per epoch
#      (visibility lebih kaya untuk debug)
#
#   3. result.json menyimpan F2_drowsy + per-class precision/recall
#
#   4. Scheduler.step() — fix bug: tanpa noise va_f1/100
# ============================================================

# ── Selection key untuk early stopping ───────────────────────
# Bisa diubah via cfg["selection_metric"] kalau mau A/B test
DEFAULT_SELECTION_METRIC = "f2_drowsy"   # ← Metrik utama skripsi


def run_tuning_experiment(
    experiment_id : str,
    backbone_name : str,
    cfg           : dict,
    skip_if_done  : bool = True,
) -> dict:
    """
    Jalankan satu eksperimen tuning end-to-end.

    Early stopping berdasarkan cfg["selection_metric"] (default: f2_drowsy).
    """
    exp_dir    = os.path.join(SAVE_DIR, experiment_id)
    os.makedirs(exp_dir, exist_ok=True)
    path_best  = os.path.join(exp_dir, f"{experiment_id}_BEST.pth")
    path_last  = os.path.join(exp_dir, f"{experiment_id}_LAST.pth")
    path_hist  = os.path.join(exp_dir, f"{experiment_id}_history.json")
    path_res   = os.path.join(exp_dir, f"{experiment_id}_result.json")

    # ── Skip jika sudah ada ──────────────────────────────────
    if skip_if_done and os.path.exists(path_res):
        with open(path_res) as f:
            result = json.load(f)
        # Backward-compat: tampilkan F2 jika tersedia, fallback F1
        f2_show = result.get("f2_drowsy", result.get("test_f1_macro", 0))
        print(f"  [SKIP] {experiment_id} — sudah ada "
              f"(F2_drowsy={f2_show:.4f})")
        return result

    lstm_type        = "BILSTM" if cfg.get("bidirectional", False) else "LSTM"
    max_epochs       = cfg.get("max_epochs", 80)
    patience         = cfg.get("patience",   15)
    batch_size       = cfg.get("batch_size", 128)
    selection_metric = cfg.get("selection_metric", DEFAULT_SELECTION_METRIC)

    print("\n" + "=" * 78)
    print(f"  EXPERIMENT: {experiment_id}")
    print(f"  Backbone={backbone_name} | {lstm_type} | "
          f"hidden={cfg.get('hidden_dim',256)} | "
          f"layers={cfg.get('num_layers',2)}")
    print(f"  LR={cfg.get('lr',1e-3):.1e} | WD={cfg.get('weight_decay',1e-4):.1e} | "
          f"act={cfg.get('fc_activation','relu')} | "
          f"sched={cfg.get('scheduler_type','cosine_warm')}")
    print(f"  Loss={cfg.get('loss_type','ce')} | "
          f"LabelSmooth={cfg.get('label_smoothing',0.0)} | "
          f"Aug={'ON' if cfg.get('use_aug',False) else 'OFF'}")
    print(f"  >>> Early stopping → {selection_metric.upper()} <<<")
    print("=" * 78)

    # ── 1. Load Data ─────────────────────────────────────────
    loaders, class_weight, norm_stats = build_dataloaders(
        backbone_name,
        batch_size   = batch_size,
        use_aug      = cfg.get("use_aug",      False),
        aug_prob_fd  = cfg.get("aug_prob_fd",  0.3),
        aug_prob_gn  = cfg.get("aug_prob_gn",  0.3),
        n_drop_max   = cfg.get("n_drop_max",   1),
        noise_std    = cfg.get("noise_std",    0.01),
    )

    # ── 2. Bangun Model ──────────────────────────────────────
    model = build_model_from_cfg(cfg)
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  [Model] Total params: {total_params:,}")

    # ── 3. Loss + Optimizer + Scheduler ─────────────────────
    criterion                      = build_loss(cfg, class_weight)
    optimizer, scheduler, sched_step_type = build_optimizer_and_scheduler(
        model, cfg, loaders
    )
    max_grad_norm = cfg.get("max_grad_norm", 1.0)

    # ── 4. Training Loop ─────────────────────────────────────
    best_score       = -1.0
    patience_counter = 0
    history = {
        "train_loss": [], "train_acc": [], "train_f1_macro": [], "train_f2_drowsy": [],
        "train_recall_drowsy": [],
        "val_loss":   [], "val_acc":   [], "val_f1_macro":   [], "val_f2_drowsy":   [],
        "val_recall_drowsy":   [], "val_precision_drowsy": [],
        "lr": [],
    }

    print(f"\n  {'Ep':>4} | {'TrLoss':>7} | {'TrF1m':>6} | {'TrF2dr':>7} | "
          f"{'VaLoss':>7} | {'VaF1m':>6} | {'VaF2dr':>7} | {'VaRecD':>7} | "
          f"{'LR':>8} | Status")
    print("  " + "─" * 100)

    t_start = time.time()

    for epoch in range(max_epochs):
        tr = train_epoch(
            model, loaders["train"], criterion, optimizer,
            scheduler     = scheduler if sched_step_type == "per_batch" else None,
            sched_type    = sched_step_type,
            max_grad_norm = max_grad_norm,
        )
        va = evaluate(model, loaders["val"], criterion)

        current_lr = optimizer.param_groups[0]["lr"]

        # ── Scheduler step per epoch (BUG FIX: tanpa noise va_f1/100) ──
        if sched_step_type == "per_epoch":
            scheduler.step(epoch + 1)
        elif sched_step_type == "per_epoch_plateau":
            scheduler.step(va[selection_metric])

        # Simpan history
        history["train_loss"].append(tr["loss"])
        history["train_acc"].append(tr["acc"])
        history["train_f1_macro"].append(tr["f1_macro"])
        history["train_f2_drowsy"].append(tr["f2_drowsy"])
        history["train_recall_drowsy"].append(tr["recall_drowsy"])
        history["val_loss"].append(va["loss"])
        history["val_acc"].append(va["acc"])
        history["val_f1_macro"].append(va["f1_macro"])
        history["val_f2_drowsy"].append(va["f2_drowsy"])
        history["val_recall_drowsy"].append(va["recall_drowsy"])
        history["val_precision_drowsy"].append(va["precision_drowsy"])
        history["lr"].append(current_lr)

        # Early stopping & save best (berdasarkan selection_metric)
        score_now = va[selection_metric]
        status = ""
        if score_now > best_score + 1e-4:
            best_score = score_now
            patience_counter = 0
            status = "★ BEST"
            torch.save({
                "model_state"  : model.state_dict(),
                "val_f1_macro" : va["f1_macro"],
                "val_f2_drowsy": va["f2_drowsy"],
                "val_recall_drowsy": va["recall_drowsy"],
                "val_precision_drowsy": va["precision_drowsy"],
                "epoch"        : epoch,
                "backbone"     : backbone_name,
                "lstm_type"    : lstm_type,
                "cfg"          : cfg,
                "norm_mean"    : norm_stats["mean"],
                "norm_std"     : norm_stats["std"],
                "selection_metric": selection_metric,
            }, path_best)
        else:
            patience_counter += 1

        print(f"  {epoch+1:>4} | {tr['loss']:>7.4f} | {tr['f1_macro']:>6.4f} | "
              f"{tr['f2_drowsy']:>7.4f} | {va['loss']:>7.4f} | "
              f"{va['f1_macro']:>6.4f} | {va['f2_drowsy']:>7.4f} | "
              f"{va['recall_drowsy']:>7.4f} | {current_lr:>8.2e} | {status}")

        # Save last checkpoint
        torch.save({
            "model_state"     : model.state_dict(),
            "optim_state"     : optimizer.state_dict(),
            "epoch"           : epoch,
            "best_score"      : best_score,
            "selection_metric": selection_metric,
            "patience_counter": patience_counter,
            "history"         : history,
            "backbone"        : backbone_name,
            "lstm_type"       : lstm_type,
            "cfg"             : cfg,
            "norm_mean"       : norm_stats["mean"],
            "norm_std"        : norm_stats["std"],
        }, path_last)

        with open(path_hist, "w") as f:
            json.dump({k: [float(v) for v in vals]
                       for k, vals in history.items()}, f, indent=2)

        if patience_counter >= patience:
            print(f"\n  → Early stopping epoch {epoch+1} "
                  f"(patience {patience} habis tanpa improve {selection_metric}).")
            break

    elapsed = time.time() - t_start
    print(f"\n  Selesai dalam {elapsed/60:.1f} menit. "
          f"Best Val {selection_metric}: {best_score:.4f}")

    # ── 5. Evaluasi Test Set ─────────────────────────────────
    print("  [TEST] Evaluasi test set dengan model BEST...")
    ckpt_best = torch.load(path_best, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt_best["model_state"])

    te = evaluate(model, loaders["test"], criterion)
    te_preds  = te["preds"]
    te_labels = te["labels"]
    te_cm     = confusion_matrix(te_labels, te_preds)

    # F1 per class (sudah dihitung), tambahkan precision/recall macro
    te_precision_macro = precision_score(te_labels, te_preds, average="macro", zero_division=0)
    te_recall_macro    = recall_score(te_labels, te_preds, average="macro", zero_division=0)
    te_f1_per_class    = f1_score(te_labels, te_preds, average=None, zero_division=0)

    # ROC-AUC dari probabilitas drowsy
    try:
        model.eval()
        all_probs_drowsy = []
        with torch.no_grad():
            for feats, _ in loaders["test"]:
                feats = feats.to(DEVICE)
                logits, _ = model(feats)
                probs = torch.softmax(logits, dim=1)[:, DROWSY_CLASS_IDX].cpu().numpy()
                all_probs_drowsy.extend(probs)
        # ROC: positif = drowsy, jadi label drowsy=1 untuk roc_auc convention
        labels_pos_drowsy = [1 if l == DROWSY_CLASS_IDX else 0 for l in te_labels]
        te_roc_auc = roc_auc_score(labels_pos_drowsy, all_probs_drowsy)
    except Exception:
        te_roc_auc = float("nan")

    # Confusion matrix breakdown (drowsy=0 di axis 0)
    true_drowsy           = int(te_cm[0, 0])
    drowsy_missed         = int(te_cm[0, 1])  # FATAL
    notdrowsy_false_alarm = int(te_cm[1, 0])
    true_notdrowsy        = int(te_cm[1, 1])

    print(f"\n  ─── HASIL TEST SET ─────────────────────────────────")
    print(f"  Accuracy         : {te['acc']:.4f}")
    print(f"  F1 Macro         : {te['f1_macro']:.4f}")
    print(f"   F2 Drowsy        : {te['f2_drowsy']:.4f}  (★ metrik utama)")
    print(f"  Precision Drowsy : {te['precision_drowsy']:.4f}")
    print(f"  Recall Drowsy    : {te['recall_drowsy']:.4f}  (safety)")
    print(f"  Precision Macro  : {te_precision_macro:.4f}")
    print(f"  Recall Macro     : {te_recall_macro:.4f}")
    print(f"  ROC-AUC (drowsy) : {te_roc_auc:.4f}")
    print(f"  F1 drowsy        : {te_f1_per_class[0]:.4f}")
    print(f"  F1 notdrowsy     : {te_f1_per_class[1]:.4f}")
    print(f"  ─── CONFUSION MATRIX ───────────────────────────────")
    print(f"  TP drowsy    : {true_drowsy}    (benar, AMAN)")
    print(f"  FN drowsy    : {drowsy_missed}  (FATAL → drowsy lewat)")
    print(f"  FP drowsy    : {notdrowsy_false_alarm}  (false alarm)")
    print(f"  TN notdrowsy : {true_notdrowsy}")

    # ── 6. Simpan Hasil ──────────────────────────────────────
    result = {
        "experiment_id"        : experiment_id,
        "backbone"             : backbone_name,
        "lstm_type"            : lstm_type,
        "selection_metric"     : selection_metric,
        "cfg"                  : {k: (v if not isinstance(v, bool) else int(v))
                                  for k, v in cfg.items()},
        "best_val_score"       : float(best_score),
        # Test metrics
        "test_acc"             : float(te["acc"]),
        "test_f1_macro"        : float(te["f1_macro"]),
        "test_f2_drowsy"       : float(te["f2_drowsy"]),    # ← BARU
        "test_precision_drowsy": float(te["precision_drowsy"]),  # ← BARU
        "test_recall_drowsy"   : float(te["recall_drowsy"]),     # ← BARU
        "test_precision_macro" : float(te_precision_macro),
        "test_recall_macro"    : float(te_recall_macro),
        "test_roc_auc"         : float(te_roc_auc),
        "f1_drowsy"            : float(te_f1_per_class[0]),
        "f1_notdrowsy"         : float(te_f1_per_class[1]),
        # Backward-compat alias (Cell 11/12 lama pakai nama ini)
        "drowsy_recall"        : float(te["recall_drowsy"]),
        # Confusion matrix
        "confusion_matrix"     : te_cm.tolist(),
        "true_drowsy"          : true_drowsy,
        "drowsy_missed"        : drowsy_missed,
        "notdrowsy_false_alarm": notdrowsy_false_alarm,
        "true_notdrowsy"       : true_notdrowsy,
        # Meta
        "elapsed_min"          : round(elapsed / 60, 2),
        "best_epoch"           : int(ckpt_best["epoch"]) + 1,
        "total_epochs_run"     : epoch + 1,
    }

    with open(path_res, "w") as f:
        json.dump(result, f, indent=2)

    print(f"\n  [SAVED] {path_res}")

    # ── 7. Cleanup GPU ───────────────────────────────────────
    del model, loaders, optimizer, scheduler, criterion
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  GPU cleared → {torch.cuda.memory_allocated()/1e6:.0f} MB allocated\n")

    return result


print("✓ CELL 8 v2 — run_tuning_experiment() terdefinisi.")
print(f"  Default selection metric: {DEFAULT_SELECTION_METRIC}")
print("  Bisa override via cfg['selection_metric'] = 'f1_macro' / 'recall_drowsy' / 'f2_drowsy'")


✓ CELL 8 v2 — run_tuning_experiment() terdefinisi.
  Default selection metric: f2_drowsy
  Bisa override via cfg['selection_metric'] = 'f1_macro' / 'recall_drowsy' / 'f2_drowsy'


# Grid Eksperimen (Semua Backbone + Semua Konfigurasi)

In [9]:
# ============================================================
# CELL 9 — EXPERIMENT GRID PERCOBAAN 3 (Round 3 Refinement)
# ============================================================
#
# STRATEGI: Exploit recipe EXP_K (pemenang Round 2).
# Hanya variasi 1 dimensi per eksperimen (ablation discipline).
#
# BASE RECIPE (EXP_K):
#   - scheduler_type = cosine_warm
#   - T_0 = 20, T_mult = 2, eta_min = 1e-6
#   - max_epochs = 120, patience = 30
#   - loss_type = ce, label_smoothing = 0.0
#   - lr = 2e-4, weight_decay = 1e-4
#   - hidden_dim = 256, num_layers = 2
#   - lstm_dropout = 0.3, fc_dropout = 0.4
#   - fc_activation = gelu (SWIN)
#   - batch_size = 128
#   - use_aug = False (Round 2 membuktikan aug hancur SWIN)
#   - bidirectional = False (Round 2 membuktikan BiLSTM kalah di SWIN)
# ============================================================

def make_k2_exp(backbone, variant_id, **overrides):
    """
    Build eksperimen Round 3 berbasis recipe EXP_K.
    backbone: 'SWIN' atau 'VGG19'
    variant_id: suffix, misal 'T30', 'LR1e4', etc.
    """
    bilstm = overrides.pop("bidirectional", False)
    lstm_tag = "BILSTM" if bilstm else "LSTM"
    exp_id = f"{backbone}_{lstm_tag}_EXP_K2_{variant_id}"

    # BASE recipe EXP_K
    cfg = dict(
        # Arsitektur
        bidirectional    = bilstm,
        use_attention    = True,
        hidden_dim       = 256,
        num_layers       = 2,
        lstm_dropout     = 0.3,
        fc_dropout       = 0.4,
        fc_activation    = "gelu" if backbone == "SWIN" else "relu",

        # Training
        batch_size       = 128,
        max_epochs       = 120,
        patience         = 30,
        max_grad_norm    = 5.0 if backbone == "SWIN" else 1.0,

        # Optimizer
        optimizer_type   = "adamw",
        lr               = 2e-4,
        weight_decay     = 1e-4,

        # Scheduler (cosine_warm long cycle)
        scheduler_type   = "cosine_warm",
        T_0              = 20,
        T_mult           = 2,
        eta_min          = 1e-6,

        # Loss
        loss_type        = "ce",
        label_smoothing  = 0.0,
        focal_gamma      = 2.0,

        # Augmentasi OFF (Round 2 bukti aug hurt SWIN)
        use_aug          = False,
        aug_prob_fd      = 0.3,
        aug_prob_gn      = 0.3,
        n_drop_max       = 1,
        noise_std        = 0.01,

        # Selection metric
        selection_metric = "f2_drowsy",
    )

    cfg.update(overrides)
    return {"experiment_id": exp_id, "backbone": backbone, "cfg": cfg}


# ─────────────────────────────────────────────────────────────
# EXPERIMENT_GRID: 11 eksperimen Round 3
# ─────────────────────────────────────────────────────────────
EXPERIMENT_GRID = [

    # ── 1. T_0=30, runway lebih panjang ─────────────────────
    # Hipotesis: EXP_K T_0=20 masih kurang → T_0=30 konvergensi lebih dalam
    make_k2_exp("SWIN", "T30",
        T_0        = 30,
        max_epochs = 150,
        patience   = 35,
    ),

    # ── 2. LR=1e-4 (half), fine-grained convergence ─────────
    # Hipotesis: LR=2e-4 masih sedikit agresif untuk recipe ini
    make_k2_exp("SWIN", "LR1e4",
        lr = 1e-4,
    ),

    # ── 3. hidden_dim=384 (kompromi 256↔512) ────────────────
    # Hipotesis: 256 bottleneck, 512 overfits → 384 sweet spot
    make_k2_exp("SWIN", "H384",
        hidden_dim = 384,
    ),

    # ── 4. hidden=512 + WD=3e-4 (selamatkan EXP_B pattern) ──
    # Hipotesis: 512 butuh WD lebih kuat (vs 1e-4 yang overfits)
    make_k2_exp("SWIN", "H512_WD3",
        hidden_dim   = 512,
        weight_decay = 3e-4,
        batch_size   = 64,   # lower batch utk VRAM
    ),

    # ── 5. Regularisasi lebih kuat ──────────────────────────
    # Hipotesis: current dropout (0.3/0.4) masih longgar
    make_k2_exp("SWIN", "DROP4",
        lstm_dropout = 0.4,
        fc_dropout   = 0.5,
    ),

    # ── 6. Focal γ=1.5 dengan runway penuh EXP_K ────────────
    # Hipotesis: EXP_I (focal γ=1.5) F2=0.65 di best_ep=5 saja.
    #   Kalau dikasih runway 120 epoch, bisa tembus EXP_K.
    make_k2_exp("SWIN", "FOCAL15",
        loss_type   = "focal",
        focal_gamma = 1.5,
    ),

    # ── 7. num_layers=3 (deeper LSTM) ───────────────────────
    # Hipotesis: 2 layer kurang untuk capture temporal drowsy pattern
    make_k2_exp("SWIN", "LAYER3",
        num_layers   = 3,
        lstm_dropout = 0.35,  # sedikit naik untuk depth
    ),

    # ── 8. BiLSTM revival: h=512 + WD=3e-4 ──────────────────
    # Hipotesis: BILSTM kalah karena WD=1e-4 kurang di param banyak
    make_k2_exp("SWIN", "BI_H512_WD3",
        bidirectional = True,
        hidden_dim    = 512,
        weight_decay  = 3e-4,
        batch_size    = 64,
    ),

    # ── 9. LinearWarmup + cosine_warm (anti-degenerate) ─────
    # Hipotesis: 5 epoch warmup smooth hindari early pathological solution
    make_k2_exp("SWIN", "WARMUP",
        scheduler_type = "warmup_cosine",
        warmup_epochs  = 5,
        T_0            = 20,
        T_mult         = 2,
    ),

    # ── 10. AGGRESSIVE: compound terbaik ────────────────────
    # Hipotesis: Compound variasi yang kemungkinan berhasil (target tertinggi)
    make_k2_exp("SWIN", "AGG",
        T_0            = 30,
        max_epochs     = 150,
        patience       = 35,
        hidden_dim     = 384,
        lstm_dropout   = 0.4,
        fc_dropout     = 0.45,
    ),

    # ── 11. VGG19 dengan recipe EXP_K ───────────────────────
    # Sanity check: apakah EXP_K recipe universal atau SWIN-specific?
    make_k2_exp("VGG19", "RECIPE",
        fc_activation = "relu",  # VGG19 pakai relu
        max_grad_norm = 1.0,
    ),
]

# ── Ringkasan Grid ────────────────────────────────────────────
df_grid = pd.DataFrame([
    {
        "Exp ID"       : e["experiment_id"],
        "Backbone"     : e["backbone"],
        "LSTM Type"    : "BiLSTM" if e["cfg"]["bidirectional"] else "LSTM",
        "Hidden"       : e["cfg"]["hidden_dim"],
        "Layers"       : e["cfg"]["num_layers"],
        "LR"           : e["cfg"]["lr"],
        "WD"           : e["cfg"]["weight_decay"],
        "Scheduler"    : e["cfg"]["scheduler_type"],
        "T_0"          : e["cfg"].get("T_0", "—"),
        "Loss"         : e["cfg"]["loss_type"],
        "γ_focal"      : e["cfg"]["focal_gamma"] if e["cfg"]["loss_type"] == "focal" else "—",
        "LS"           : e["cfg"]["label_smoothing"],
        "Dropout"      : f"{e['cfg']['lstm_dropout']}/{e['cfg']['fc_dropout']}",
        "Aug"          : "ON" if e["cfg"]["use_aug"] else "OFF",
        "Batch"        : e["cfg"]["batch_size"],
        "MaxEp"        : e["cfg"]["max_epochs"],
    }
    for e in EXPERIMENT_GRID
])

print(f"✓ Total eksperimen Round 3: {len(EXPERIMENT_GRID)}")
print()
print("  Daftar Eksperimen:")
print(df_grid.to_string(index=False))
print()
print("   ESTIMASI WAKTU (RTX 3060, features pre-extracted):")
print(f"    ~4-5 menit × {len(EXPERIMENT_GRID)} = ~40-55 menit total")
print()
print("   TARGET: F2 Drowsy ≥ 0.70 (current best: 0.668 dari EXP_K)")
print("   Setelah selesai, jalankan CELL 10 → CELL 11 → CELL 12")


✓ Total eksperimen Round 3: 11

  Daftar Eksperimen:
                        Exp ID Backbone LSTM Type  Hidden  Layers     LR     WD     Scheduler  T_0  Loss γ_focal  LS  Dropout Aug  Batch  MaxEp
          SWIN_LSTM_EXP_K2_T30     SWIN      LSTM     256       2 0.0002 0.0001   cosine_warm   30    ce       — 0.0  0.3/0.4 OFF    128    150
        SWIN_LSTM_EXP_K2_LR1e4     SWIN      LSTM     256       2 0.0001 0.0001   cosine_warm   20    ce       — 0.0  0.3/0.4 OFF    128    120
         SWIN_LSTM_EXP_K2_H384     SWIN      LSTM     384       2 0.0002 0.0001   cosine_warm   20    ce       — 0.0  0.3/0.4 OFF    128    120
     SWIN_LSTM_EXP_K2_H512_WD3     SWIN      LSTM     512       2 0.0002 0.0003   cosine_warm   20    ce       — 0.0  0.3/0.4 OFF     64    120
        SWIN_LSTM_EXP_K2_DROP4     SWIN      LSTM     256       2 0.0002 0.0001   cosine_warm   20    ce       — 0.0  0.4/0.5 OFF    128    120
      SWIN_LSTM_EXP_K2_FOCAL15     SWIN      LSTM     256       2 0.0002 0.0001   c

# Eksekusi Semua Eksperimen (Sequential, Skip jika Done)

In [10]:
# ============================================================
# CELL 10 — EKSEKUSI SEMUA EKSPERIMEN (Percobaan 3)
# ============================================================
# Loop otomatis melalui semua entry di EXPERIMENT_GRID.
# - skip_if_done=True → lanjutkan dari checkpoint, tidak ulang
# - Filter BACKBONE_FILTER untuk menjalankan sebagian saja
#
# REKOMENDASI: Untuk Round 3, biarkan filter None karena hanya
# 11 eksperimen total (~45-55 menit).
# ============================================================

# ── Konfigurasi Eksekusi ──────────────────────────────────────
BACKBONE_FILTER = None   # None = semua, atau ["SWIN"] / ["VGG19", "SWIN"]
EXP_GROUP_FILTER = None  # None = semua, atau ["K2_AGG", "K2_T30"]

ALL_TUNING_RESULTS = []

# ── Filter grid ───────────────────────────────────────────────
grid_to_run = EXPERIMENT_GRID
if BACKBONE_FILTER is not None:
    grid_to_run = [e for e in grid_to_run if e["backbone"] in BACKBONE_FILTER]
if EXP_GROUP_FILTER is not None:
    grid_to_run = [
        e for e in grid_to_run
        if any(g in e["experiment_id"] for g in EXP_GROUP_FILTER)
    ]

print(f"[EKSEKUSI] Menjalankan {len(grid_to_run)} dari {len(EXPERIMENT_GRID)} eksperimen")
print(f"  Filter backbone  : {BACKBONE_FILTER or 'SEMUA'}")
print(f"  Filter exp group : {EXP_GROUP_FILTER or 'SEMUA'}")
print(f"  skip_if_done     : True")
print()

t_total_start = time.time()

# ── Helper: aman ambil kolom dari result dict ────────────────
def _safe_progress_row(r):
    """Build 1 baris CSV progress, robust terhadap missing key."""
    return {
        "experiment_id"   : r.get("experiment_id", "?"),
        "backbone"        : r.get("backbone", "?"),
        "lstm_type"       : r.get("lstm_type", "?"),
        "test_acc"        : r.get("test_acc", 0),
        "test_f2_drowsy"  : r.get("test_f2_drowsy", 0),
        "test_f1_macro"   : r.get("test_f1_macro", 0),
        "f1_drowsy"       : r.get("f1_drowsy", 0),
        "f1_notdrowsy"    : r.get("f1_notdrowsy", 0),
        "drowsy_recall"   : r.get("drowsy_recall",
                                  r.get("test_recall_drowsy", 0)),
        "test_precision_drowsy": r.get("test_precision_drowsy", 0),
        "test_roc_auc"    : r.get("test_roc_auc", 0),
        "drowsy_missed"   : r.get("drowsy_missed", 0),
        "best_val_score"  : r.get("best_val_score",
                                  r.get("best_val_f1", 0)),  # backward-compat
        "best_epoch"      : r.get("best_epoch", 0),
        "elapsed_min"     : r.get("elapsed_min", 0),
    }

for i, exp_entry in enumerate(grid_to_run):
    print(f"\n[{i+1}/{len(grid_to_run)}] " + "─" * 50)
    result = run_tuning_experiment(
        experiment_id = exp_entry["experiment_id"],
        backbone_name = exp_entry["backbone"],
        cfg           = exp_entry["cfg"],
        skip_if_done  = True,
    )
    ALL_TUNING_RESULTS.append(result)

    # Auto-save progress CSV setelah setiap eksperimen
    rows = [_safe_progress_row(r) for r in ALL_TUNING_RESULTS]
    df_progress = pd.DataFrame(rows).sort_values(
        "test_f2_drowsy", ascending=False
    )
    csv_path = os.path.join(SAVE_DIR, "PROGRESS_RESULTS_p3.csv")
    df_progress.to_csv(csv_path, index=False)

t_total_elapsed = time.time() - t_total_start
print(f"\n{'='*70}")
print(f"  SEMUA EKSPERIMEN SELESAI dalam {t_total_elapsed/3600:.2f} jam "
      f"({t_total_elapsed/60:.1f} menit)")
print(f"  Progress tersimpan: {csv_path}")
print(f"{'='*70}")
print()
print("  → Lanjutkan ke CELL 11 untuk RANKING + DEGENERATE FLAG")
print("  → Lalu CELL 12 untuk SAFETY WINNER")


[EKSEKUSI] Menjalankan 11 dari 11 eksperimen
  Filter backbone  : SEMUA
  Filter exp group : SEMUA
  skip_if_done     : True


[1/11] ──────────────────────────────────────────────────

  EXPERIMENT: SWIN_LSTM_EXP_K2_T30
  Backbone=SWIN | LSTM | hidden=256 | layers=2
  LR=2.0e-04 | WD=1.0e-04 | act=gelu | sched=cosine_warm
  Loss=ce | LabelSmooth=0.0 | Aug=OFF
  >>> Early stopping → F2_DROWSY <<<
  [DataLoader] SWIN | batch=128 | aug=OFF
  [Norm]       mean=0.2107  std=0.2008
  [ClassW]     drowsy=0.8944  notdrowsy=1.1339
  [Model] Total params: 1,415,042

    Ep |  TrLoss |  TrF1m |  TrF2dr |  VaLoss |  VaF1m |  VaF2dr |  VaRecD |       LR | Status
  ────────────────────────────────────────────────────────────────────────────────────────────────────
     1 |  0.5621 | 0.6730 |  0.6309 |  1.1014 | 0.5628 |  0.4915 |  0.4595 | 2.00e-04 | ★ BEST
     2 |  0.3902 | 0.7944 |  0.8181 |  1.4696 | 0.5418 |  0.5441 |  0.5271 | 1.99e-04 | ★ BEST
     3 |  0.3278 | 0.8405 |  0.8521 |  1.5486 | 0

# Tabel Kesimpulan Ranking & Rekomendasi Implementasi

In [11]:
# ============================================================
# CELL 11 — KESIMPULAN RANKING + DEGENERATE DETECTION (Percobaan 3)
# ============================================================
# PERUBAHAN:
#   + Flag "DEGENERATE" (best_epoch ≤ 3 AND precision_drowsy < 0.48)
#     → Model tetap muncul di ranking untuk perbandingan skripsi,
#       tapi ditandai agar tidak disalahpilih sebagai pemenang.
#   + Ranking menunjukkan P2 + P3 bersamaan (gabungan).
# ============================================================

print("=" * 80)
print("  CELL 11 Percobaan 3 — RANKING + DEGENERATE FLAG")
print("=" * 80)

import os
import json
import pandas as pd

# ── Threshold DEGENERATE detection ───────────────────────────
DEGEN_MAX_EPOCH      = 3      # best_epoch ≤ ini
DEGEN_MIN_PRECISION  = 0.48   # precision_drowsy < ini

def is_degenerate(row):
    """Model degenerate = model yang 'beruntung' di early epoch tanpa belajar."""
    try:
        be = int(row.get("best_epoch", 99))
        pd_ = float(row.get("test_precision_drowsy",
                           row.get("precision_drowsy", 1.0)))
    except Exception:
        return False
    return (be <= DEGEN_MAX_EPOCH) and (pd_ < DEGEN_MIN_PRECISION)


loaded_results = []
for exp_folder in sorted(os.listdir(SAVE_DIR)):
    res_path = os.path.join(SAVE_DIR, exp_folder, f"{exp_folder}_result.json")
    if os.path.exists(res_path):
        with open(res_path) as f:
            result = json.load(f)

            f2_drowsy_val = result.get("test_f2_drowsy", None)
            if f2_drowsy_val is None:
                f2_drowsy_val = result.get("f1_drowsy", 0)

            row = {
                "experiment_id"        : result["experiment_id"],
                "backbone"             : result["backbone"],
                "lstm_type"            : result.get("lstm_type", "?"),
                "test_acc"             : result["test_acc"],
                "test_f1_macro"        : result["test_f1_macro"],
                "test_f2_drowsy"       : f2_drowsy_val,
                "test_recall_drowsy"   : result.get("test_recall_drowsy",
                                                   result.get("drowsy_recall", 0)),
                "test_precision_drowsy": result.get("test_precision_drowsy", 0),
                "f1_drowsy"            : result.get("f1_drowsy", 0),
                "f1_notdrowsy"         : result.get("f1_notdrowsy", 0),
                "drowsy_recall"        : result.get("drowsy_recall",
                                                  result.get("test_recall_drowsy", 0)),
                "test_roc_auc"         : result.get("test_roc_auc", 0),
                "drowsy_missed"        : result.get("drowsy_missed", 999),
                "best_epoch"           : result.get("best_epoch", 0),
                "lr"                   : result["cfg"].get("lr", 0) if "cfg" in result else 0,
                "weight_decay"         : result["cfg"].get("weight_decay", 0) if "cfg" in result else 0,
                "scheduler_type"       : result["cfg"].get("scheduler_type", "?") if "cfg" in result else "?",
                "loss_type"            : result["cfg"].get("loss_type", "?") if "cfg" in result else "?",
                "focal_gamma"          : result["cfg"].get("focal_gamma", 0) if "cfg" in result else 0,
                "fc_activation"        : result["cfg"].get("fc_activation", "?") if "cfg" in result else "?",
                "hidden_dim"           : result["cfg"].get("hidden_dim", 0) if "cfg" in result else 0,
                "num_layers"           : result["cfg"].get("num_layers", 2) if "cfg" in result else 2,
                "lstm_dropout"         : result["cfg"].get("lstm_dropout", 0) if "cfg" in result else 0,
                "fc_dropout"           : result["cfg"].get("fc_dropout", 0) if "cfg" in result else 0,
                "batch_size"           : result["cfg"].get("batch_size", 0) if "cfg" in result else 0,
                "use_aug"              : result["cfg"].get("use_aug", False) if "cfg" in result else False,
                "label_smoothing"      : result["cfg"].get("label_smoothing", 0) if "cfg" in result else 0,
                "optimizer_type"       : result["cfg"].get("optimizer_type", "?") if "cfg" in result else "?",
                "T_0"                  : result["cfg"].get("T_0", "—") if "cfg" in result else "—",
            }
            row["is_degenerate"] = is_degenerate(row)
            loaded_results.append(row)

if not loaded_results:
    print("⚠ Belum ada hasil eksperimen. Jalankan CELL 10 terlebih dahulu.")
else:
    df_all = pd.DataFrame(loaded_results)
    n_degen = df_all["is_degenerate"].sum()

    print(f"\n  Total eksperimen: {len(df_all)}")
    print(f"  Degenerate (best_ep ≤ {DEGEN_MAX_EPOCH} AND prec_drowsy < {DEGEN_MIN_PRECISION}): {n_degen}")
    print()

    # ── Ranking gabungan (semua) dengan flag ─────────────────
    df_ranked = df_all.sort_values(
        by=["test_f2_drowsy", "drowsy_missed"],
        ascending=[False, True]
    ).reset_index(drop=True)
    df_ranked.index = df_ranked.index + 1

    # ── Print Top-20 ─────────────────────────────────────────
    print(" TOP-20 RANKING — F2 DROWSY (semua, gabungan P2+P3)")
    print("─" * 150)
    header = (f"{'Rank':>4} | {'Experiment ID':<38} | {'Acc':>6} | "
              f"{'F2dr':>6} | {'F1Mac':>6} | {'RecDr':>6} | {'PreDr':>6} | "
              f"{'AUC':>6} | {'FNdro':>6} | {'BEp':>4} | Flag")
    print(header)
    print("─" * 150)
    for rank, row in df_ranked.head(20).iterrows():
        prefix = "★" if "SWIN" in row["experiment_id"] else " "
        flag = " DEGEN" if row["is_degenerate"] else ""
        k2_mark = " K2" if "K2" in row["experiment_id"] else ""
        try:
            be = int(row["best_epoch"])
        except Exception:
            be = 0
        print(f"{prefix}{rank:>4} | {row['experiment_id']:<38} | "
              f"{row['test_acc']:>6.4f} | {float(row['test_f2_drowsy']):>6.4f} | "
              f"{row['test_f1_macro']:>6.4f} | {float(row['drowsy_recall']):>6.4f} | "
              f"{float(row['test_precision_drowsy']):>6.4f} | "
              f"{float(row['test_roc_auc']):>6.4f} | "
              f"{int(row['drowsy_missed']):>6} | "
              f"{be:>4} |{flag}{k2_mark}")
    print("─" * 150)
    print("  ★ = SWIN   |   F2dr = F2-Score Drowsy (utama)   |   BEp = Best Epoch")
    print("  ⚠ DEGEN = best_epoch rendah + precision rendah = kemungkinan trivial bias")
    print("   K2 = Round 3 (Percobaan 3)")

    # ── Ranking HANYA legitimate (non-degenerate) ────────────
    df_legit = df_ranked[~df_ranked["is_degenerate"]].reset_index(drop=True)
    df_legit.index += 1

    print("\n\n TOP-15 LEGIT MODELS (non-degenerate only)")
    print("─" * 150)
    print(header)
    print("─" * 150)
    for rank, row in df_legit.head(15).iterrows():
        prefix = "★" if "SWIN" in row["experiment_id"] else " "
        k2_mark = " K2" if "K2" in row["experiment_id"] else ""
        try:
            be = int(row["best_epoch"])
        except Exception:
            be = 0
        print(f"{prefix}{rank:>4} | {row['experiment_id']:<38} | "
              f"{row['test_acc']:>6.4f} | {float(row['test_f2_drowsy']):>6.4f} | "
              f"{row['test_f1_macro']:>6.4f} | {float(row['drowsy_recall']):>6.4f} | "
              f"{float(row['test_precision_drowsy']):>6.4f} | "
              f"{float(row['test_roc_auc']):>6.4f} | "
              f"{int(row['drowsy_missed']):>6} | "
              f"{be:>4} |{k2_mark}")
    print("─" * 150)

    # ── Focus: Round 3 results only ──────────────────────────
    df_k2 = df_ranked[df_ranked["experiment_id"].str.contains("K2")]
    if not df_k2.empty:
        print("\n\n HASIL ROUND 3 (Percobaan 3) — SAJA")
        print("─" * 150)
        print(f"{'Rank':>4} | {'Experiment ID':<38} | {'F2dr':>6} | "
              f"{'F1Mac':>6} | {'RecDr':>6} | {'PreDr':>6} | "
              f"{'FNdro':>6} | {'BEp':>4} | Flag")
        print("─" * 150)
        for rank, row in df_k2.sort_values("test_f2_drowsy", ascending=False).iterrows():
            flag = " DEGEN" if row["is_degenerate"] else ""
            try:
                be = int(row["best_epoch"])
            except Exception:
                be = 0
            print(f"{rank:>4} | {row['experiment_id']:<38} | "
                  f"{float(row['test_f2_drowsy']):>6.4f} | "
                  f"{row['test_f1_macro']:>6.4f} | "
                  f"{float(row['drowsy_recall']):>6.4f} | "
                  f"{float(row['test_precision_drowsy']):>6.4f} | "
                  f"{int(row['drowsy_missed']):>6} | "
                  f"{be:>4} |{flag}")

    # ── Best per backbone (legit only) ───────────────────────
    print("\n\n KONFIGURASI TERBAIK PER BACKBONE (LEGIT only, by F2)")
    print("─" * 100)
    for backbone in ["CNN_BASIC", "MOBILENET", "VGG19", "SWIN"]:
        df_bb_legit = df_legit[df_legit["backbone"] == backbone]
        if not df_bb_legit.empty:
            best = df_bb_legit.iloc[0]
            print(f"\n  [{backbone}]  (legit only)")
            print(f"    Experiment ID    : {best['experiment_id']}")
            print(f"    F2 Drowsy        : {float(best['test_f2_drowsy']):.4f}  (utama)")
            print(f"    F1 Macro         : {best['test_f1_macro']:.4f}")
            print(f"    Accuracy         : {best['test_acc']:.4f}")
            print(f"    Recall Drowsy    : {float(best['drowsy_recall']):.4f}")
            print(f"    Precision Drowsy : {float(best['test_precision_drowsy']):.4f}")
            print(f"    FN Drowsy        : {int(best['drowsy_missed'])}")
            print(f"    Best Epoch       : {int(best['best_epoch'])}")
            print(f"    LR / WD          : {best['lr']} / {best['weight_decay']}")
            print(f"    Scheduler / T_0  : {best['scheduler_type']} / {best.get('T_0','—')}")
            print(f"    Loss / LS        : {best['loss_type']} / {best['label_smoothing']}")
            print(f"    Hidden / Layers  : {best['hidden_dim']} / {best['num_layers']}")

    # ── Save CSV ─────────────────────────────────────────────
    summary_path = os.path.join(SAVE_DIR, "SUMMARY_ALL_EXPERIMENTS_p3.csv")
    df_ranked.to_csv(summary_path, index=True)
    print(f"\n  [SAVED] {summary_path}")


  CELL 11 Percobaan 3 — RANKING + DEGENERATE FLAG

  Total eksperimen: 11
  Degenerate (best_ep ≤ 3 AND prec_drowsy < 0.48): 0

 TOP-20 RANKING — F2 DROWSY (semua, gabungan P2+P3)
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Rank | Experiment ID                          |    Acc |   F2dr |  F1Mac |  RecDr |  PreDr |    AUC |  FNdro |  BEp | Flag
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
★   1 | SWIN_LSTM_EXP_K2_H512_WD3              | 0.5691 | 0.6141 | 0.5690 | 0.6551 | 0.4911 | 0.5826 |    189 |    4 | K2
★   2 | SWIN_LSTM_EXP_K2_LR1e4                 | 0.5922 | 0.5752 | 0.5886 | 0.5931 | 0.5134 | 0.5780 |    223 |   25 | K2
★   3 | SWIN_LSTM_EXP_K2_AGG                   | 0.5768 | 0.5698 | 0.5741 | 0.5912 | 0.4977 | 0.5776 |    224 |   49 | K2
★   4 | SWIN_BILSTM_EXP_K2_BI

# Safety-Weighted Winner: Kriteria Keselamatan Skripsi

In [12]:
# ============================================================
# CELL 12 — SAFETY WINNER (Percobaan 3 — Exclude Degenerate)
# ============================================================
# PERUBAHAN:
#   + Kandidat safety HARUS non-degenerate (is_degenerate=False)
#   + Selain itu sama dengan P2 (filter + safety score formula)
# ============================================================

print("=" * 80)
print("  CELL 12 Percobaan 3 — SAFETY-WEIGHTED WINNER (non-degenerate only)")
print("=" * 80)

# ── Parameter Ambang Keselamatan ─────────────────────────────
SAFETY_MAX_MISSED   = 150
SAFETY_MIN_RECALL   = 0.70
ACADEMIC_MIN_F1     = 0.60   # slight relaxation dari P2 (0.65)
MIN_F2_DROWSY       = 0.65

TOTAL_DROWSY_TEST = 548

def compute_safety_score(row):
    """Safety Score = 0.40×F2 + 0.25×F1 + 0.20×Recall + 0.15×(1-FN%)"""
    try:
        f2_drowsy = float(row.get("test_f2_drowsy", 0))
    except Exception:
        f2_drowsy = 0
    try:
        f1_macro  = float(row.get("test_f1_macro",  0))
    except Exception:
        f1_macro = 0
    try:
        drowsy_recall = float(row.get("drowsy_recall",
                                       row.get("test_recall_drowsy", 0)))
    except Exception:
        drowsy_recall = 0
    try:
        missed_n = float(row.get("drowsy_missed", TOTAL_DROWSY_TEST))
    except Exception:
        missed_n = TOTAL_DROWSY_TEST
    missed_ratio   = missed_n / TOTAL_DROWSY_TEST
    missed_penalty = 1.0 - missed_ratio

    score = (0.40 * f2_drowsy) + (0.25 * f1_macro) + \
            (0.20 * drowsy_recall) + (0.15 * missed_penalty)
    return round(score, 6)


try:
    _ = df_ranked
    df_eval = df_ranked.copy()
    print("  [INFO] Menggunakan df_ranked dari CELL 11.")
except NameError:
    csv_path = os.path.join(SAVE_DIR, "SUMMARY_ALL_EXPERIMENTS_p3.csv")
    if os.path.exists(csv_path):
        df_eval = pd.read_csv(csv_path)
        print(f"  [INFO] Loaded dari {csv_path}")
    else:
        raise RuntimeError("Run CELL 11 dahulu.")

# Pastikan kolom numerik
for col in ["test_f2_drowsy", "drowsy_recall", "drowsy_missed",
            "test_precision_drowsy", "test_roc_auc", "test_f1_macro",
            "test_acc"]:
    if col in df_eval.columns:
        df_eval[col] = pd.to_numeric(df_eval[col], errors="coerce").fillna(0)

# ── Hitung safety score ──────────────────────────────────────
df_eval["safety_score"] = df_eval.apply(compute_safety_score, axis=1)

# ── RANKING LEGIT saja ────────────────────────────────────────
df_legit = df_eval[~df_eval["is_degenerate"]].copy()
print(f"\n  Legit models (non-degenerate): {len(df_legit)}")

# ── Filter safety ────────────────────────────────────────────
df_safe = df_legit[
    (df_legit["drowsy_missed"]   <  SAFETY_MAX_MISSED) &
    (df_legit["drowsy_recall"]   >= SAFETY_MIN_RECALL) &
    (df_legit["test_f1_macro"]   >= ACADEMIC_MIN_F1) &
    (df_legit["test_f2_drowsy"]  >= MIN_F2_DROWSY)
].sort_values(
    ["safety_score", "test_f2_drowsy"],
    ascending=[False, False]
).reset_index(drop=True)
df_safe.index += 1


# ── TAMPILAN RANKING LEGIT F2 ─────────────────────────────────
print("\n┌" + "─" * 78 + "┐")
print("│  [A] RANKING LEGIT F2 DROWSY (non-degenerate, utama)                │")
print("└" + "─" * 78 + "┘")
df_legit_f2 = df_legit.sort_values("test_f2_drowsy", ascending=False).reset_index(drop=True)
df_legit_f2.index += 1
print(f"  {'Rank':>4} │ {'Experiment ID':<38} │ {'F2dr':>6} │ {'F1Mac':>6} │ "
      f"{'RecDr':>6} │ {'PreDr':>6} │ {'FNdro':>5} │ {'BEp':>4}")
print("  " + "─" * 4 + "─┼─" + "─" * 38 + "─┼─" + "─" * 6 + "─┼─" + "─" * 6 +
      "─┼─" + "─" * 6 + "─┼─" + "─" * 6 + "─┼─" + "─" * 5 + "─┼─" + "─" * 4)
for rank, row in df_legit_f2.head(10).iterrows():
    flag = " " if "SWIN" in str(row.get("experiment_id", "")) else "  "
    medal = "" if rank == 1 else ("" if rank == 2 else ("" if rank == 3 else f"{rank:>3} "))
    try:
        be = int(row["best_epoch"])
    except Exception:
        be = 0
    print(f"  {medal}{flag} │ {str(row.get('experiment_id', '?')):<38} │ "
          f"{row.get('test_f2_drowsy', 0):>6.4f} │ "
          f"{row.get('test_f1_macro', 0):>6.4f} │ "
          f"{row.get('drowsy_recall', 0):>6.4f} │ "
          f"{row.get('test_precision_drowsy', 0):>6.4f} │ "
          f"{int(row.get('drowsy_missed', 0)):>5} │ "
          f"{be:>4}")

if not df_legit_f2.empty:
    winner_f2 = df_legit_f2.iloc[0]
    print(f"\n   PEMENANG LEGIT F2 : {winner_f2['experiment_id']}")
    print(f"     F2 Drowsy           : {winner_f2['test_f2_drowsy']:.4f}")
    print(f"     F1 Macro            : {winner_f2['test_f1_macro']:.4f}")
    print(f"     FN Drowsy           : {int(winner_f2['drowsy_missed'])}")
    print(f"     Best Epoch          : {int(winner_f2['best_epoch'])}")


# ── TAMPILAN SAFETY (ketat) ──────────────────────────────────
print("\n\n┌" + "─" * 78 + "┐")
print("│  [B] RANKING KESELAMATAN — Legit + Filter Ketat                     │")
print(f"│  Syarat: FN<{SAFETY_MAX_MISSED} │ RecDr≥{SAFETY_MIN_RECALL} │ "
      f"F1Mac≥{ACADEMIC_MIN_F1} │ F2dr≥{MIN_F2_DROWSY}  │ non-degen        │")
print(f"│  Score = 0.40×F2 + 0.25×F1 + 0.20×Recall + 0.15×(1-FN%)            │")
print("└" + "─" * 78 + "┘")

if df_safe.empty:
    print("\n  ⚠ Tidak ada model yang memenuhi semua syarat.")
    print("\n  Kandidat terdekat (legit saja):")
    df_close = df_legit[df_legit["test_f1_macro"] >= 0.55].sort_values(
        "safety_score", ascending=False
    ).head(10)
    for rank_c, (_, row_c) in enumerate(df_close.iterrows(), 1):
        missed_ok = "✓" if row_c["drowsy_missed"] < SAFETY_MAX_MISSED else "✗"
        recall_ok = "✓" if row_c["drowsy_recall"] >= SAFETY_MIN_RECALL else "✗"
        f2_ok     = "✓" if row_c["test_f2_drowsy"] >= MIN_F2_DROWSY else "✗"
        f1_ok     = "✓" if row_c["test_f1_macro"] >= ACADEMIC_MIN_F1 else "✗"
        try:
            be = int(row_c["best_epoch"])
        except Exception:
            be = 0
        print(f"  {rank_c:>2}. {str(row_c.get('experiment_id','?')):<40} │ "
              f"F2={row_c['test_f2_drowsy']:.4f}{f2_ok} │ "
              f"F1={row_c['test_f1_macro']:.4f}{f1_ok} │ "
              f"FN={int(row_c['drowsy_missed'])}{missed_ok} │ "
              f"Rec={row_c['drowsy_recall']:.4f}{recall_ok} │ "
              f"BEp={be} │ "
              f"Safe={row_c['safety_score']:.4f}")
else:
    print(f"\n  ✓ {len(df_safe)} model lolos semua syarat keselamatan:\n")
    print(f"  {'Rank':>4} │ {'Experiment ID':<38} │ {'Safe':>9} │ "
          f"{'F2dr':>6} │ {'F1Mac':>6} │ {'RecDr':>6} │ {'FNdro':>5}")
    for rank_s, row_s in df_safe.head(10).iterrows():
        flag = " " if "SWIN" in str(row_s.get("experiment_id", "")) else "  "
        print(f"  {rank_s:>4}{flag} │ {str(row_s.get('experiment_id', '?')):<38} │ "
              f"{row_s['safety_score']:>9.6f} │ "
              f"{row_s.get('test_f2_drowsy', 0):>6.4f} │ "
              f"{row_s.get('test_f1_macro', 0):>6.4f} │ "
              f"{row_s.get('drowsy_recall', 0):>6.4f} │ "
              f"{int(row_s.get('drowsy_missed', 0)):>5}")

    winner = df_safe.iloc[0]
    print(f"\n   PEMENANG KESELAMATAN : {winner['experiment_id']}")
    print(f"     Safety Score          : {winner['safety_score']:.6f}")
    print(f"     F2 Drowsy             : {winner['test_f2_drowsy']:.4f}")
    print(f"     F1 Macro              : {winner['test_f1_macro']:.4f}")
    print(f"     Recall Drowsy         : {winner['drowsy_recall']:.4f}")
    print(f"     FN Drowsy             : {int(winner['drowsy_missed'])}")

# ── Save ─────────────────────────────────────────────────────
safety_csv_path = os.path.join(SAVE_DIR, "SAFETY_RANKING_p3.csv")
df_eval.sort_values("safety_score", ascending=False).to_csv(
    safety_csv_path, index=False
)
print(f"\n  [SAVED] {safety_csv_path}")
print("=" * 80)
print("  ✓ CELL 12 Percobaan 3 selesai.")
print("=" * 80)


  CELL 12 Percobaan 3 — SAFETY-WEIGHTED WINNER (non-degenerate only)
  [INFO] Menggunakan df_ranked dari CELL 11.

  Legit models (non-degenerate): 11

┌──────────────────────────────────────────────────────────────────────────────┐
│  [A] RANKING LEGIT F2 DROWSY (non-degenerate, utama)                │
└──────────────────────────────────────────────────────────────────────────────┘
  Rank │ Experiment ID                          │   F2dr │  F1Mac │  RecDr │  PreDr │ FNdro │  BEp
  ─────┼────────────────────────────────────────┼────────┼────────┼────────┼────────┼───────┼─────
    │ SWIN_LSTM_EXP_K2_H512_WD3              │ 0.6141 │ 0.5690 │ 0.6551 │ 0.4911 │   189 │    4
    │ SWIN_LSTM_EXP_K2_LR1e4                 │ 0.5752 │ 0.5886 │ 0.5931 │ 0.5134 │   223 │   25
    │ SWIN_LSTM_EXP_K2_AGG                   │ 0.5698 │ 0.5741 │ 0.5912 │ 0.4977 │   224 │   49
    4   │ SWIN_BILSTM_EXP_K2_BI_H512_WD3         │ 0.5670 │ 0.5030 │ 0.6131 │ 0.4358 │   212 │   31
    5   │ SWIN_LSTM_EXP_K2_D

##  REKOMENDASI POST-ROUND-3

### Interpretasi Hasil

1. **Jika EXP_K2_AGG atau EXP_K2_T30 menang dengan F2 ≥ 0.70**:
   → Pindah ke implementasi `inference.py` dengan config tersebut
   → Skripsi argument: "Long-cycle cosine_warm (T_0=30) superior untuk feature-level LSTM"

2. **Jika EXP_K2_FOCAL15 menang**:
   → Novelty angle kuat: "Soft focal loss (γ=1.5) + runway panjang mengatasi
     keterbatasan focal loss tradisional (γ=2) di dataset label-noise"

3. **Jika EXP_K2_H384 atau EXP_K2_H512_WD3 menang**:
   → Capacity-regularization tradeoff diperlukan
   → Hidden sweet spot ditemukan

4. **Jika tidak ada yang kalahkan EXP_K**:
   → EXP_K adalah ceiling dari recipe ini
   → Perlu explorasi diluar LSTM: Transformer head, atau partial Swin unfreeze

### Langkah Selanjutnya (Week 1 → Week 2)

**Hari 3 (sekarang)**:
- Run 11 eksperimen Round 3 (~45-55 menit)
- Analisis hasil di CELL 11 & 12
- Kirim hasil ke mentor untuk keputusan final model

**Hari 4-5**: Implementasi ke `inference.py`
- Load model pemenang
- Update MIN_DROWSY_CONFIDENCE berdasarkan ROC curve optimal threshold
- Validasi live webcam EMEET C60E

**Hari 6-7**: Dokumentasi
- Bab 4: Engineering findings (bug _eval_alert, degenerate detection)
- Bab 3: Novelty hyperparameter ablation 89 eksperimen total (P2: 78 + P3: 11)

**Hari 8-14**: Penulisan Bab + preparasi sidang

### Safeguard yang Sudah Ditambahkan

1. **DEGENERATE flag** — otomatis mendeteksi model yang converge di epoch 1-3
   dengan precision rendah (trivial recall-bias solution)
2. **Safety filter ketat** — safety winner hanya dari non-degenerate pool
3. **Ranking gabungan P2+P3** — semua hasil sejarah bisa dibandingkan


# Membuat norm_stats.npz

In [13]:
import os
import numpy as np
import torch

#  Hardcode langsung — tidak bergantung variabel dari cell lain
SEQ_TRAIN_PATH = r"C:\kuliah-sementara\SKRIPSI\sequences_nthuddd2\SWIN\SWIN_Seq_Train_NTHUD.pt"
SAVE_PATH      = r"C:\kuliah-sementara\SKRIPSI\streamlit_app\models\norm_stats.npz"

print(f"[INFO] Membaca: {SEQ_TRAIN_PATH}")
data     = torch.load(SEQ_TRAIN_PATH, map_location="cpu", weights_only=False)
features = data["features"].float()

N, T, D  = features.shape
flat     = features.reshape(-1, D)
mean     = flat.mean(dim=0).numpy()
std      = flat.std(dim=0).clamp(min=1e-8).numpy()

print(f"[INFO] Shape: {list(features.shape)}")
print(f"[INFO] mean range: [{mean.min():.4f}, {mean.max():.4f}]")
print(f"[INFO] std  range: [{std.min():.6f}, {std.max():.4f}]")

os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
np.savez(SAVE_PATH, mean=mean, std=std)

print(f"\n[OK] Tersimpan ke: {SAVE_PATH}")

[INFO] Membaca: C:\kuliah-sementara\SKRIPSI\sequences_nthuddd2\SWIN\SWIN_Seq_Train_NTHUD.pt
[INFO] Shape: [8053, 30, 512]
[INFO] mean range: [0.0000, 1.7790]
[INFO] std  range: [0.000000, 0.9395]

[OK] Tersimpan ke: C:\kuliah-sementara\SKRIPSI\streamlit_app\models\norm_stats.npz


# Fungsi untuk menganalisis kurva Learning Rate vs Loss 

In [1]:
def analyze_lr_loss(lrs, losses, smoothing_weight=0.8):
    """
    Fungsi untuk menganalisis kurva Learning Rate vs Loss (biasanya dari LR Finder) 
    untuk mengidentifikasi fase underfitting, optimal (plateau), dan overfitting.
    
    Parameter:
    lrs: List atau array dari learning rates.
    losses: List atau array dari nilai test loss yang bersesuaian.
    smoothing_weight: Bobot untuk menghaluskan kurva loss (Membantu mengurangi noise).
    
    Return:
    Dictionary yang berisi status analisis dan batas maximum LR.
    """
    import numpy as np
    
    # 1. Menghaluskan loss untuk analisis yang lebih baik (Mengurangi noise)
    smoothed_losses = []
    for loss in losses:
        if not smoothed_losses:
            smoothed_losses.append(loss)
        else:
            smoothed_losses.append(smoothing_weight * smoothed_losses[-1] + (1 - smoothing_weight) * loss)
            
    smoothed_losses = np.array(smoothed_losses)
    lrs = np.array(lrs)
    
    # Menghitung gradient (perubahan loss)
    gradients = np.gradient(smoothed_losses)
    
    # Mencari titik minimum loss
    min_loss_idx = np.argmin(smoothed_losses)
    min_loss = smoothed_losses[min_loss_idx]
    best_lr_at_min = lrs[min_loss_idx]
    
    # Analisis Fase
    # Fase 1: Underfitting - Loss terus menurun dengan stabil (gradient negatif beruntun)
    # Fase 2: Plateau (Tujuan Utama) - Loss stabil di nilai rendah / horizontal (gradient mendekati 0)
    # Fase 3: Overfitting - Loss mulai meningkat kembali (gradient positif)
    
    # 4. Hasil: Batas LR Maksimum (Titik dimana loss berhenti menurun dan mulai meningkat/meroket)
    # Mencari titik di mana loss mulai "meroket" atau naik secara signifikan setelah titik minimum
    # Kita cari indeks setelah min_loss di mana gradient positifnya konsisten dan membesar
    max_lr_idx = min_loss_idx
    for i in range(min_loss_idx, len(smoothed_losses) - 1):
        if gradients[i] > 0 and gradients[i+1] > 0: # Cek jika mulai naik
            # Batas akhirnya adalah titik tepat sebelum mulai naik
            max_lr_idx = i
            break
            
    max_lr = lrs[max_lr_idx]
    
    # Cek kesimpulan umum dari kurva
    is_underfitting = np.all(gradients < 0) # Jika dari awal sampai akhir turun terus
    is_overfitting = len(smoothed_losses) - min_loss_idx > (0.2 * len(smoothed_losses)) and gradients[-1] > 0 # Jika ujungnya naik signifikan
    
    print("=== Analisis Evaluasi Kurva LR vs Loss ===")
    print(f"1. Underfitting: Kurva menunjukkan loss {'turun terus menerus' if is_underfitting else 'mencapai batas bawah'}.")
    print(f"2. Overfitting: Kurva menunjukkan test loss {'mulai meningkat' if is_overfitting else 'tidak meningkat secara signifikan'} setelah titik minimum.")
    print(f"3. Tujuan Utama (Plateau/Keseimbangan): Terjadi di sekitar rentang LR {lrs[max(0, min_loss_idx-5)]:.2E} sampai {best_lr_at_min:.2E}")
    print(f"4. Hasil (Batas LR Maksimum): Terdeteksi pada LR = {max_lr:.2E} (Titik di mana loss berhenti menurun dan mulai naik)")
    
    return {
        "min_loss": min_loss,
        "best_lr_at_min": best_lr_at_min,
        "max_usable_lr": max_lr,
        "is_underfitting_only": is_underfitting,
        "is_overfitting_detected": is_overfitting
    }